# 📂 Data Versioning with DVC (Data Version Control)

This notebook demonstrates:

1. 📝 **Create a sample dataset** (review data)
2. 📁 **Create `data/` folder** and save the dataset as CSV
3. ☁️ **Create `S3/` folder** (simulating S3 remote storage)
4. 🔧 **Initialize DVC** and track the data folder
5. 🔄 **Version the data** with DVC push/pull

---

In [3]:
import os
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

---
## Step 1: Create a Dictionary, Convert it to a Pandas DataFrame

In [5]:
def create_dataframe():
    data = {
        "id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "review": [
            "Great food and ambiance.",
            "Terrible service.",
            "Amazing experience!",
            "Food was cold.",
            "Loved the desserts.",
            "Not worth the money.",
            "Excellent customer service.",
            "The place was too crowded.",
            "Best restaurant in town.",
            "Average experience."
        ]
    }
    df = pd.DataFrame(data)
    return df

df = create_dataframe()
print(f"DataFrame shape: {df.shape}")
df

DataFrame shape: (10, 2)


,id,review
0,1,Great food and ambiance.
1,2,Terrible service.
2,3,Amazing experience!
3,4,Food was cold.
4,5,Loved the desserts.
5,6,Not worth the money.
6,7,Excellent customer service.
7,8,The place was too crowded.
8,9,Best restaurant in town.
9,10,Average experience.


---
## Step 2: Check if `data` Folder Exists, Create if Not, and Save as `data.csv`

In [6]:
def save_dataframe(df):
    if not os.path.exists("data"):
        os.makedirs("data")
    df.to_csv("data/data.csv", index=False)
    print("data.csv saved in 'data' folder.")

save_dataframe(df)

# Verify the file was saved
print(f"\nFiles in 'data/' folder: {os.listdir('data')}")
print(f"File size: {os.path.getsize('data/data.csv')} bytes")

data.csv saved in 'data' folder.

Files in 'data/' folder: ['data.csv']
File size: 250 bytes


---
## Step 3: Create `S3/` Folder (Simulating S3 Remote Storage)

In a real-world scenario, you would use an actual AWS S3 bucket.  
Here we simulate S3 with a **local folder** acting as a DVC remote.

In [7]:
def create_s3_folder():
    s3_path = "S3"
    if not os.path.exists(s3_path):
        os.makedirs(s3_path)
        print(f"✅ '{s3_path}/' folder created (simulating S3 remote storage).")
    else:
        print(f"ℹ️  '{s3_path}/' folder already exists.")
    return os.path.abspath(s3_path)

s3_abs_path = create_s3_folder()
print(f"S3 folder absolute path: {s3_abs_path}")

✅ 'S3/' folder created (simulating S3 remote storage).
S3 folder absolute path: /Users/jogeswararao/ML_AI_TRAINING_SESSIONS/CODE_DATA_MODEL_VERSIONING_IN_ML_LIFECYCLE/S3


---
## Step 4: Feature Engineering with CountVectorizer

In [ ]:
def process_data(df):
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(df['review'])
    feature_names = vectorizer.get_feature_names_out()
    
    # Create a DataFrame with the vectorized features
    features_df = pd.DataFrame(X.toarray(), columns=feature_names)
    result_df = pd.concat([df, features_df], axis=1)
    
    # Save the processed data
    result_df.to_csv("data/processed_data.csv", index=False)
    print(f"Processed data saved: data/processed_data.csv")
    print(f"Features extracted: {len(feature_names)}")
    print(f"Feature names: {list(feature_names)}")
    
    return result_df

processed_df = process_data(df)
print(f"\nProcessed DataFrame shape: {processed_df.shape}")
processed_df

---
## Step 5: Initialize DVC & Track Data

Run these commands in the terminal to set up DVC:

```text
DVC Workflow:

    data/data.csv ──► dvc add ──► data/data.csv.dvc (metadata)
         │                              │
         ▼                              ▼
    .dvc/cache (local)            Git tracks .dvc file
         │
         ▼
    dvc push ──► S3/ (remote storage)
```

In [ ]:
# ──────────────────────────────────────────────
# Step 5a: Initialize DVC in the project
# ──────────────────────────────────────────────
!dvc init

In [ ]:
# ──────────────────────────────────────────────
# Step 5b: Add the local S3 folder as DVC remote
# ──────────────────────────────────────────────
!dvc remote add -d cloud_bucket S3/

In [ ]:
# ──────────────────────────────────────────────
# Step 5c: Track the data folder with DVC
# ──────────────────────────────────────────────
!dvc add data/data.csv
!dvc add data/processed_data.csv

In [ ]:
# ──────────────────────────────────────────────
# Step 5d: Git commit the .dvc tracking files
# ──────────────────────────────────────────────
!git add data/data.csv.dvc data/processed_data.csv.dvc data/.gitignore .dvc/config
!git commit -m "Track data files with DVC v1"

In [ ]:
# ──────────────────────────────────────────────
# Step 5e: Push data to remote (S3/ folder)
# ──────────────────────────────────────────────
!dvc push
print("\n✅ Data pushed to remote storage (S3/ folder)")
print(f"\nFiles in S3/ remote:")
for root, dirs, files in os.walk('S3'):
    for f in files:
        filepath = os.path.join(root, f)
        print(f"  {filepath} ({os.path.getsize(filepath)} bytes)")

---
## Step 6: Verify DVC Setup

In [ ]:
# ──────────────────────────────────────────────
# Verify DVC tracking status
# ──────────────────────────────────────────────
print("=" * 60)
print("📋 DVC STATUS")
print("=" * 60)
!dvc status

print("\n" + "=" * 60)
print("📂 DVC REMOTE LIST")
print("=" * 60)
!dvc remote list

print("\n" + "=" * 60)
print("📄 DVC TRACKING FILES (.dvc)")
print("=" * 60)
for f in os.listdir('data'):
    if f.endswith('.dvc'):
        print(f"\n── {f} ──")
        with open(os.path.join('data', f), 'r') as fh:
            print(fh.read())

---
## Step 7: Simulate Data Change & Version 2

Let's modify the data, re-track with DVC, and push a new version.

In [ ]:
# ──────────────────────────────────────────────
# Add new reviews to simulate data change
# ──────────────────────────────────────────────
new_reviews = pd.DataFrame({
    "id": [11, 12, 13],
    "review": [
        "Absolutely wonderful dining experience!",
        "The wait time was too long.",
        "Fresh ingredients and great flavors."
    ]
})

# Load existing data and append
df_v2 = pd.concat([df, new_reviews], ignore_index=True)
df_v2.to_csv("data/data.csv", index=False)

# Re-process with CountVectorizer
processed_v2 = process_data(df_v2)

print(f"\n✅ Data updated! New shape: {df_v2.shape}")
print(f"Added {len(new_reviews)} new reviews")
df_v2.tail(5)

In [ ]:
# ──────────────────────────────────────────────
# Re-track and push Version 2
# ──────────────────────────────────────────────
!dvc add data/data.csv
!dvc add data/processed_data.csv
!git add data/data.csv.dvc data/processed_data.csv.dvc
!git commit -m "Track data files with DVC v2 - added 3 new reviews"
!dvc push

print("\n✅ Version 2 pushed to remote!")

---
## Step 8: DVC Pull — Retrieve Data from Remote

If you delete the local data, you can restore it from the remote using `dvc pull`.

In [ ]:
# ──────────────────────────────────────────────
# Simulate data loss and recovery
# ──────────────────────────────────────────────

# Remove local data file
# os.remove("data/data.csv")
# print("❌ data/data.csv deleted!")

# Recover from DVC remote
# !dvc pull
# print("✅ Data restored from remote!")

print("ℹ️  Uncomment the lines above to test dvc pull (data recovery from remote).")
print("\n── How to switch between data versions ──")
print("  git log --oneline          # See commits")
print("  git checkout <commit_hash> # Switch to a version")
print("  dvc checkout               # Restore data for that version")

---
## 📋 DVC Commands Reference

| Command | Description |
|---|---|
| `dvc init` | Initialize DVC in the project |
| `dvc remote add -d myremote S3/` | Add a remote storage (local S3 folder) |
| `dvc add data/data.csv` | Start tracking a file with DVC |
| `dvc push` | Push tracked data to the remote |
| `dvc pull` | Pull data from the remote |
| `dvc status` | Check tracking status |
| `dvc checkout` | Restore data to match the current `.dvc` file |
| `dvc remote list` | List configured remotes |
| `dvc diff` | Show changes between versions |

### Versioning Workflow

```text
    ┌──────────┐     dvc add      ┌──────────────┐     git add/commit    ┌─────────┐
    │ data.csv │ ───────────────►  │ data.csv.dvc │ ──────────────────►   │   Git   │
    └──────────┘                   └──────────────┘                       └─────────┘
         │                                                                     │
         ▼                                                                     ▼
    .dvc/cache                                                          Version History
         │                                                                     │
         ▼  dvc push                                                           │
    ┌──────────┐                                        git checkout ◄──────────┘
    │  S3/     │                                              │
    │ (remote) │ ◄─────── dvc pull ◄──────── dvc checkout ◄───┘
    └──────────┘
```